# Geospatial Machine Learning

## Overview

Geospatial ML integrates geographic features — coordinates, distances, spatial relationships, and raster-extracted covariates — into predictive models. Spatial data violates the i.i.d. assumption of standard ML, requiring specialised cross-validation and evaluation strategies.

**Key considerations:**

| Challenge | Problem | Solution |
|---|---|---|
| Spatial autocorrelation | Nearby samples not independent | Spatial cross-validation |
| Coordinate leakage | Raw coords as features | Use spatial features carefully |
| Extrapolation | Predictions beyond sample space | Check covariate coverage |
| Cluster sampling | Data concentrated in accessible areas | Block CV, sample weighting |
| Scale mismatch | Features at wrong resolution | Multi-scale feature extraction |

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)

# Generate spatially autocorrelated dataset
n = 200
x = rng.uniform(317000, 327000, n)
y = rng.uniform(731000, 741000, n)
# Covariate features (raster-derived)
elevation  = 50 + 300*(1 - (y-731000)/10000) - 80*np.exp(-((x-322000)**2)/4e6) + rng.normal(0,8,n)
slope      = np.abs(rng.normal(8, 4, n)).clip(0, 35)
ndvi       = (0.7 - 0.001*(elevation-50)/3.5 + rng.normal(0, 0.05, n)).clip(0.1, 0.9)
dist_river = np.sqrt((x-322000)**2)/500 + rng.uniform(0, 2, n)
# Target: species richness with spatial + environmental drivers
richness = (25 - 0.02*elevation + 3*ndvi - 0.1*slope - 0.5*dist_river
            + rng.normal(0, 2, n))
richness = richness.clip(3).round(1)
df = pd.DataFrame({'x':x,'y':y,'elevation':elevation,'slope':slope,
                    'ndvi':ndvi,'dist_river':dist_river,'richness':richness})
print(f"Dataset: {n} sites, richness range: {richness.min():.1f}–{richness.max():.1f}")

---
## Feature Engineering from Spatial Context

In [ ]:
# Feature engineering: neighbourhood aggregation at multiple radii
coords = np.column_stack([x, y])
D = cdist(coords, coords)

def neighbourhood_mean(values, D, radius):
    means = []
    for i in range(len(values)):
        mask = (D[i] > 0) & (D[i] <= radius)
        means.append(values[mask].mean() if mask.sum() > 0 else values[i])
    return np.array(means)

# 1km and 3km neighbourhood means
df['neigh_ndvi_1km']  = neighbourhood_mean(ndvi,      D, 1000)
df['neigh_elev_1km']  = neighbourhood_mean(elevation, D, 1000)
df['neigh_ndvi_3km']  = neighbourhood_mean(ndvi,      D, 3000)
df['neigh_rich_1km']  = neighbourhood_mean(richness,  D, 1000)   # use carefully!
df['site_density_2km'] = [(D[i] <= 2000).sum() - 1 for i in range(n)]
print("Added spatial context features:")
for col in ['neigh_ndvi_1km','neigh_elev_1km','neigh_ndvi_3km','neigh_rich_1km','site_density_2km']:
    print(f"  {col:22s}: mean={df[col].mean():.3f}")

---
## Spatial vs Random Cross-Validation

In [ ]:
# Standard random k-fold CV (ignores spatial autocorrelation)
features_base = ['elevation','slope','ndvi','dist_river']
features_full = features_base + ['neigh_ndvi_1km','neigh_elev_1km','neigh_ndvi_3km','site_density_2km']

X_base = df[features_base].values
X_full = df[features_full].values
y_vec  = df['richness'].values

scaler = StandardScaler()
X_base_s = scaler.fit_transform(X_base)
X_full_s = StandardScaler().fit_transform(X_full)

rf = RandomForestRegressor(n_estimators=200, random_state=42)

# Random CV
r2_rand = cross_val_score(rf, X_full_s, y_vec, cv=KFold(10, shuffle=True, random_state=42),
                            scoring='r2').mean()

# Spatial block CV: divide into spatial blocks, hold out entire blocks
def spatial_block_cv(X, y, coords, n_folds=5, seed=42):
    r = np.random.default_rng(seed)
    # K-means spatial blocks
    from sklearn.cluster import KMeans
    km = KMeans(n_clusters=n_folds, random_state=seed, n_init=10)
    block_labels = km.fit_predict(coords)
    scores = []
    for fold in range(n_folds):
        test_mask  = block_labels == fold
        train_mask = ~test_mask
        if train_mask.sum() < 10: continue
        sc = StandardScaler()
        Xtr = sc.fit_transform(X[train_mask])
        Xte = sc.transform(X[test_mask])
        model = RandomForestRegressor(n_estimators=200, random_state=42)
        model.fit(Xtr, y[train_mask])
        scores.append(r2_score(y[test_mask], model.predict(Xte)))
    return np.mean(scores)

r2_spatial = spatial_block_cv(X_full, y_vec, coords, n_folds=5)
print(f"Random k-fold CV R²:       {r2_rand:.3f}  (optimistically biased)")
print(f"Spatial block CV R²:       {r2_spatial:.3f}  (more realistic estimate)")
print(f"Optimism gap:              {r2_rand - r2_spatial:.3f}")
print("\nSpatial CV is more honest: nearby samples in train & test inflate random CV R²")

---
## Model Comparison and Spatial Prediction Surface

In [ ]:
# Fit and compare models with spatial block CV
models = {
    'Ridge':    Ridge(alpha=1.0),
    'RF (base features)': RandomForestRegressor(200, random_state=42),
    'RF (spatial features)': RandomForestRegressor(200, random_state=42),
    'GBM':      GradientBoostingRegressor(200, random_state=42),
}
Xs = {'Ridge': X_full_s,
      'RF (base features)': X_base_s,
      'RF (spatial features)': X_full_s,
      'GBM': X_full_s}
print(f"{'Model':28s} {'Block CV R²':>12} {'Random CV R²':>13}")
for name, model in models.items():
    X_m = Xs[name]
    r2_b = spatial_block_cv(X_m, y_vec, coords)
    r2_r = cross_val_score(model, X_m, y_vec,
                            cv=KFold(10, shuffle=True, random_state=42),
                            scoring='r2').mean()
    print(f"  {name:26s} {r2_b:12.3f} {r2_r:13.3f}")

In [ ]:
# Prediction surface: fit on all data, predict on grid
best_model = RandomForestRegressor(300, random_state=42)
sc_final   = StandardScaler()
X_fit = sc_final.fit_transform(X_full)
best_model.fit(X_fit, y_vec)

# Grid prediction (using mean neighbourhood features for grid points)
gx_pred = np.linspace(317000, 327000, 60)
gy_pred = np.linspace(731000, 741000, 60)
GX, GY = np.meshgrid(gx_pred, gy_pred)
grid_pts = np.column_stack([GX.ravel(), GY.ravel()])
elev_g  = 50 + 300*(1-(GY.ravel()-731000)/10000) - 80*np.exp(-((GX.ravel()-322000)**2)/4e6)
ndvi_g  = (0.7 - 0.001*(elev_g-50)/3.5).clip(0.1, 0.9)
slope_g = np.full(len(grid_pts), slope.mean())
dr_g    = np.sqrt((GX.ravel()-322000)**2)/500
X_grid  = np.column_stack([elev_g, slope_g, ndvi_g, dr_g,
                             ndvi_g, elev_g, ndvi_g, np.full(len(grid_pts), 5)])
X_grid_s = sc_final.transform(X_grid)
y_grid   = best_model.predict(X_grid_s).reshape(GX.shape)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im = axes[0].contourf(GX, GY, y_grid, levels=20, cmap='YlGn')
axes[0].scatter(x, y, c=richness, cmap='YlGn', s=20, edgecolors='black',
                linewidths=0.3, vmin=richness.min(), vmax=richness.max(), zorder=3)
plt.colorbar(im, ax=axes[0], label='Predicted richness', shrink=0.8)
axes[0].set_title('Predicted Richness Surface (RF)')
axes[0].set_xlabel('Easting'); axes[0].set_ylabel('Northing')
# Feature importance
fi = best_model.feature_importances_
feat_names = features_full
fi_df = pd.Series(fi, index=feat_names).sort_values()
fi_df.plot.barh(ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Feature Importances'); axes[1].set_xlabel('Importance')
plt.tight_layout(); plt.show()

---

## Common Pitfalls

**1. Using random cross-validation for spatially autocorrelated data**  
When training and test samples are geographically nearby, they share similar feature values due to spatial autocorrelation. Random k-fold CV places nearby points in both folds, producing optimistically biased R² estimates that can be 0.1–0.3 higher than true out-of-sample performance. Always use spatial block or leave-location-out CV for geospatial data.

**2. Including raw coordinates (x, y) as model features without understanding the implications**  
Raw coordinates are perfect proxies for spatial autocorrelation — the model learns the spatial trend rather than the underlying ecological process. Predictions at new locations outside the training area will be unreliable. If spatial trend is meaningful, use it explicitly as a spatial covariate (e.g. northing as a proxy for latitude effect) and interpret the coefficient accordingly.

**3. Using neighbourhood-aggregated response variable as a predictor**  
Including neighbourhood mean richness as a feature when predicting richness creates a circular dependency — the feature is computed from the thing you are trying to predict. This inflates R² dramatically (especially in random CV) but produces a model that cannot be applied to new sites without known richness values.

**4. Not checking whether prediction locations fall within the environmental covariate space of training data**  
A model may predict confidently at locations where the combination of environmental covariates (e.g. high elevation + high NDVI) never appeared in training data. This is extrapolation, not interpolation. Always visualise training data covariate ranges and flag predictions outside the convex hull of the training feature space.

**5. Choosing spatial block size arbitrarily without checking for residual spatial autocorrelation**  
If spatial blocks are smaller than the range of spatial autocorrelation, nearby blocks will share similar residuals and the spatial CV will still be optimistic. Choose block size to exceed the variogram range, and verify that residuals from the chosen CV are not spatially autocorrelated (check Moran's I on CV residuals).
---
*python_methods_library - Samantha McGarrigle*